<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Correspondencias de nomenclaturas arancelarias y clasificación industrial

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Conversión de códigos HS de 2017 a 2012

Carga la tabla oficial de correspondencia entre la nomenclatura HS 2017 y la HS 2012.

**Pasos realizados:**

1. **Estandarización de códigos:** renombra las columnas a `HS6` (HS 2012) y `HS6_17` (HS 2017) y las completa a seis dígitos con `zfill`.

2. **Exportación:** guarda el resultado en formato parquet para evitar releer el archivo de Excel en los pasos siguientes.

In [3]:
df_hs2012_2017 = pd.read_excel(raw / 'comercio' / 'correlativas' / 'HS2017toHS2012ConversionAndCorrelationTables.xlsx')
df_hs2012_2017.rename(columns={'To HS 2012': 'HS6','From HS 2017':'HS6_17'}, inplace=True)
df_hs2012_2017['HS6'] = df_hs2012_2017['HS6'].astype(str).str.zfill(6)
df_hs2012_2017['HS6_17'] = df_hs2012_2017['HS6_17'].astype(str).str.zfill(6)

temp.mkdir(parents=True, exist_ok=True)
df_hs2012_2017.to_parquet(temp / 'df_hs2012_2017.parquet', index=False)

### Conversión de códigos HS de 2022 a 2012

Carga la tabla de correspondencia entre la nomenclatura HS 2022 y la HS 2012, con el mismo
tratamiento aplicado a la tabla de 2017.

In [4]:
df_hs2012_2022 = pd.read_excel(raw / 'comercio' / 'correlativas' / 'HS2022toHS2012ConversionAndCorrelationTables.xlsx')
df_hs2012_2022.rename(columns={'To HS 2012': 'HS6','From HS 2022':'HS6_22'}, inplace=True)
df_hs2012_2022['HS6'] = df_hs2012_2022['HS6'].astype(str).str.zfill(6)
df_hs2012_2022['HS6_22'] = df_hs2012_2022['HS6_22'].astype(str).str.zfill(6)

temp.mkdir(parents=True, exist_ok=True)
df_hs2012_2022.to_parquet(temp / 'df_hs2012_2022.parquet', index=False)

### Clasificación Leamer por código HS6

Asigna a cada código HS 2012 una de las categorías de Leamer.

**Pasos realizados:**

1. **Correspondencia SITC a Leamer:** parte del archivo corregido que asocia cada código SITC de dos dígitos a una categoría.

2. **Cruce HS a SITC:** enlaza cada HS6 con su SITC de dos dígitos mediante la tabla de correlación HS 2012 a SITC Rev. 4.

3. **Selección de la categoría dominante:** cuando un HS6 queda vinculado a varias categorías, conserva la de mayor frecuencia.

4. **Traducción de etiquetas:** aplica el diccionario de traducción al español de las categorías.

5. **Complemento de faltantes:** incorpora las asignaciones manuales para los códigos sin correspondencia tras el cruce.

In [5]:
df_sitc_leamer = pd.read_excel(raw / 'comercio' / 'correlativas' / 'FIXEDLeamer HS 2012 to SITC Rev.4 Correlation and conversion tables 2.xls', sheet_name='Sheet1').rename(columns={'SITC 2 Fixed':'SITC','Leamer_category':'Leamer'})
df_sitc_leamer['SITC'] = df_sitc_leamer['SITC'].astype(str).str.zfill(2)
df_sitc_leamer = df_sitc_leamer[['SITC','Leamer']].drop_duplicates()

df_sitc_hs = pd.read_excel(raw / 'comercio' / 'correlativas' / 'Leamer HS 2012 to SITC Rev.4 Correlation and conversion tables (1).xls', sheet_name='Correlation_HS12-SITC4', header=1)
df_sitc_hs['HS6'] = df_sitc_hs['HS 2012.1'].str.replace('.','').str.replace('HS ','').astype(str).str.zfill(6)
df_sitc_hs['SITC'] = df_sitc_hs['SITC Rev. 4'].astype(str).str[:2]
df_sitc_hs = df_sitc_hs[['HS6','SITC']].drop_duplicates()

df_sitc_hs = df_sitc_hs.merge(df_sitc_leamer, how='left', on='SITC').groupby(['HS6','Leamer']).count().reset_index()
df_sitc_hs['SITC max'] = df_sitc_hs.groupby(['HS6']).transform('max')['SITC']

df_leamer = df_sitc_hs.query("SITC == `SITC max`").drop(columns=['SITC max','SITC'])
traduccion_leamer = {
    'Animal Agriculture': 'Agricultura animal',
    'Animal agriculture': 'Agricultura animal',
    'Tropical Agriculture': 'Agricultura tropical',
    'Cereals': 'Cereales',
    'Chemicals': 'Químicos',
    'Labor Intensive': 'Intensivo en mano de obra',
    'Capital Intensive': 'Intensivo en capital',
    'Raw Materials': 'Materias primas',
    'Petroleum': 'Petróleo',
    'Machinery': 'Maquinaria',
    'Forest Products': 'Productos forestales'
}

df_leamer['Leamer'] = df_leamer['Leamer'].replace(traduccion_leamer)
df_missing_leamer = pd.read_excel(raw / 'comercio' / 'correlativas' / 'HS Missing Leamer - Fixed.xlsx')[['HS6', 'Leamer']]
df_missing_leamer['HS6'] = df_missing_leamer['HS6'].astype(str).str.zfill(6)
df_leamer = pd.concat([df_leamer, df_missing_leamer], ignore_index=True).drop_duplicates()

temp.mkdir(parents=True, exist_ok=True)
df_leamer.to_parquet(temp / 'df_leamer.parquet', index=False)

### Clasificación BEC

Traduce al español la variable `Type` de la clasificación por grandes categorías económicas:
formación bruta de capital fijo, consumo intermedio y consumo final.

In [6]:
df_BEC = pd.read_parquet(raw / 'comercio' / 'df_BEC.parquet')
traduccion_bec = {
    'Gross Fixed Capital Formation': 'Formación Bruta de Capital Fijo',
    'Intermediate Consumption': 'Consumo Intermedio',
    'Final Consumption': 'Consumo Final'
    }
df_BEC['Type'] = df_BEC['Type'].replace(traduccion_bec)

temp.mkdir(parents=True, exist_ok=True)
df_BEC.to_parquet(temp / 'df_BEC.parquet', index=False)

### Nombres de los productos HS6

Prepara las descripciones de los códigos HS 2012.

**Pasos realizados:**

1. **Estandarización del código:** completa el código HS6 a seis dígitos.

2. **Extracción del nombre principal:** conserva el primer segmento de `Description`, anterior al primer punto y coma.

In [7]:
df_hs_names = pd.read_csv(raw / 'comercio' / 'correlativas' / 'HS6Names2012.csv')
df_hs_names['HS6'] = df_hs_names['HS6'].astype(str).str.zfill(6)
df_hs_names['Name'] = df_hs_names['Description'].str.split(';').str[0]

temp.mkdir(parents=True, exist_ok=True)
df_hs_names.to_parquet(temp / 'df_hs_names.parquet', index=False)

### Correspondencia entre subpartidas arancelarias e industrias CIIU

Vincula cada código arancelario de seis dígitos con un sector económico de la CIIU Rev. 4 A.C.

El proyecto requiere esta correspondencia en dos nomenclaturas distintas: la cadena de
caracterización trabaja en HS 2012 y la de impacto en HS 1996. El procedimiento es el mismo
en ambos casos y solo cambia el año de corte, por lo que se define una función y se invoca
con la tabla de parámetros correspondiente.

**Pasos realizados dentro de la función:**

1. **Filtro temporal:** conserva las observaciones del archivo DIRPEN hasta el año de corte.

2. **Selección del registro más reciente:** para cada subpartida toma el año máximo dentro del rango.

3. **Estandarización de códigos:** lleva la subpartida a seis dígitos y el código CIIU a cuatro.

4. **Incorporación de la descripción:** cruza con la estructura detallada de la CIIU y conserva un registro por HS6.

In [8]:
def construir_ciiu(anio_corte):
    df_totpart = pd.read_excel(raw / 'ciiu' / 'anex-DIRPEN-TOTPARTv76-ago2024.xlsx', header=2)
    # EXCEPCION DE MIGRACION: el archivo DIRPEN se actualizo el 4-ago-2025, un dia despues
    # de la ultima corrida del notebook original, y quedo con un valor 'z' en la subpartida
    # 0102100000. Sin esta coercion la comparacion str <= int aborta y el codigo no corre.
    # Verificado: con la coercion los dos parquets salen identicos a los originales.
    df_totpart['Año'] = pd.to_numeric(df_totpart['Año'], errors='coerce')
    df_totpart = df_totpart[df_totpart['Año'] <= anio_corte]
    idx = df_totpart.groupby('Subpartida Arancelaria')['Año'].idxmax()
    df_totpart = df_totpart[['id','Subpartida Arancelaria','CIIU Rev. 4 A.C (2022)']].loc[idx].dropna().sort_values(['id'], ascending=True).reset_index(drop=True)
    df_totpart['Subpartida Arancelaria'] = df_totpart['Subpartida Arancelaria'].astype(str).str.zfill(10)
    df_totpart['Subpartida Arancelaria'] = df_totpart['Subpartida Arancelaria'].str[0:6]
    df_totpart['CIIU Rev. 4 A.C (2022)'] = df_totpart['CIIU Rev. 4 A.C (2022)'].astype(int).astype(str).str.zfill(4)
    df_totpart = df_totpart.drop_duplicates(['Subpartida Arancelaria','CIIU Rev. 4 A.C (2022)']).reset_index(drop=True).drop(columns=['id'], axis=1).rename(columns={'Subpartida Arancelaria':'HS6','CIIU Rev. 4 A.C (2022)':'CIIU'})

    df_estructura = pd.read_excel(raw / 'ciiu' / 'EstructuraDetalladaCIIU_4AC.xls', header=2).drop(columns=['Grupo','División']).rename(columns={'Clase':'CIIU','Descripción':'Industria'}).dropna()
    df_estructura['CIIU'] = df_estructura['CIIU'].astype(int).astype(str).str.zfill(4)

    return df_totpart.merge(df_estructura, on='CIIU', how='left').groupby('HS6').first().reset_index()


# tabla de parámetros: año de corte y archivo de salida
parametros_ciiu = [
    (2012, 'df_ciiu.parquet'),       # nomenclatura HS 2012, cadena de caracterización
    (1996, 'df_ciiu_1996.parquet'),  # nomenclatura HS 1996, cadena de impacto
]

temp.mkdir(parents=True, exist_ok=True)
for anio_corte, salida in parametros_ciiu:
    df_ciiu = construir_ciiu(anio_corte)
    df_ciiu.to_parquet(temp / salida, index=False)
    print(f"{salida}: {df_ciiu.shape[0]:,} filas")

df_ciiu.parquet: 6,352 filas


df_ciiu_1996.parquet: 5,440 filas
